# VieNeu-TTS Colab Notebook (Có UI)
Notebook tham khảo từ [VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS) với giao diện Gradio thân thiện cho phép nhập Text và Clone giọng trực tiếp.

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install "transformers==4.57.6"
!pip install vieneu gradio

In [ ]:
import gradio as gr
from vieneu import Vieneu
import os

print("Đang tải mô hình VieNeu-TTS...")
# Khởi tạo mô hình (Tự động nhận diện GPU nếu có)
vieneu = Vieneu()

# Lấy danh sách giọng dựng sẵn
try:
    voices_list = vieneu.list_preset_voices()
    voice_choices = [label for label, voice_id in voices_list]
except:
    voice_choices = ["Trúc Ly", "Minh Đức", "Ngọc Huyền", "Tuyên"]

if not voice_choices:
    voice_choices = ["Trúc Ly"]

def process_tts(text, voice_choice, ref_audio):
    if not text.strip():
        return None, "Vui lòng nhập văn bản."
    
    output_path = "output.wav"
    
    # Xử lý Clone Giọng (nếu người dùng có tải lên âm thanh mẫu)
    if ref_audio is not None and os.path.exists(ref_audio):
        try:
            print(f"Đang clone giọng từ {ref_audio}...")
            audio = vieneu.infer(
                text=text, 
                ref_audio=ref_audio, 
                denoise=True, 
                style="tu_nhien"
            )
            vieneu.save(audio, output_path)
            return output_path, "Thành công! Đã sinh giọng nói bằng tính năng Clone."
        except Exception as e:
            return None, f"Lỗi clone giọng: {str(e)}"
    else:
        # Sử dụng giọng dựng sẵn
        try:
            print(f"Đang sinh giọng với preset {voice_choice}...")
            audio = vieneu.infer(text=text, voice=voice_choice, style="tu_nhien")
            vieneu.save(audio, output_path)
            return output_path, f"Thành công! Đã sinh giọng nói với giọng {voice_choice}."
        except Exception as e:
            return None, f"Lỗi sinh giọng: {str(e)}"

# Xây dựng giao diện Gradio
with gr.Blocks(title="VieNeu-TTS Colab UI") as app:
    gr.Markdown("# 🎙️ VieNeu-TTS - Google Colab UI")
    gr.Markdown("Công cụ tổng hợp và nhân bản giọng nói (Zero-shot Voice Cloning) tiếng Việt.")
    
    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="Văn bản đầu vào", 
                lines=6, 
                placeholder="Nhập văn bản tiếng Việt cần chuyển thành giọng nói..."
            )
            
            voice_dropdown = gr.Dropdown(
                choices=voice_choices, 
                value=voice_choices[0], 
                label="Chọn giọng dựng sẵn (Preset Voice)"
            )
            
            gr.Markdown("### 🎤 Tính năng Clone Giọng (Tùy chọn)")
            gr.Markdown("Nếu bạn muốn nhân bản giọng nói, hãy tải lên một đoạn âm thanh mẫu (dài 3-8 giây, rõ chữ, ít ồn). **Lưu ý**: Nếu bạn tải lên file mẫu, hệ thống sẽ ưu tiên dùng Clone Giọng và bỏ qua lựa chọn Preset Voice ở trên.")
            ref_audio_input = gr.Audio(type="filepath", label="Âm thanh mẫu (Audio Reference)")
            
            generate_btn = gr.Button("🚀 Sinh Giọng Nói", variant="primary")
            
        with gr.Column():
            gr.Markdown("### 🎧 Kết quả")
            audio_output = gr.Audio(label="Audio")
            status_output = gr.Textbox(label="Trạng thái", interactive=False)
            
    generate_btn.click(
        fn=process_tts,
        inputs=[text_input, voice_dropdown, ref_audio_input],
        outputs=[audio_output, status_output]
    )

# Chạy ứng dụng trên Colab
app.launch(debug=True, inline=True)
